# CREACIÓN DE RED NEURONAL PARA GENERAR EMBEDDINGS CON TENSORFLOW

# paso 1 - importamos librerias

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# paso 2 - Dataset de ejemplo

In [2]:
sentences = [
    "I love deep learning",
    "I love machine learning",
    "deep learning is fun"
]

# paso 3 - Tokenizar las frases (convertir las palabras a índices)

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

# paso 4 - creamos el vocabulario

In [4]:
vocab_size = len(tokenizer.word_index) + 1  # Añadimos 1 para considerar el índice 0 (pad)
print("Vocabulary size:", vocab_size)
tokenizer.word_index

Vocabulary size: 8


{'learning': 1, 'i': 2, 'love': 3, 'deep': 4, 'machine': 5, 'is': 6, 'fun': 7}

# paso 5 - Convertir las frases a secuencias de índices

In [5]:
sequences = tokenizer.texts_to_sequences(sentences)

# Pad sequences para asegurar que todas las secuencias tengan la misma longitud

In [6]:
max_length = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

print("Padded sequences:", padded_sequences)

Padded sequences: [[2 3 4 1]
 [2 3 5 1]
 [4 1 6 7]]


# CREAMOS EL MODELO DE RED NEURONAL PARA EMBEDDING

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

In [8]:
# Definir el modelo
embedding_dim = 8  # Dimensión de los embeddings (cada palabra será representada por un vector de 8 números)

model = Sequential()

# Capa de Embeddings
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))

# Capa de GlobalAveragePooling1D para hacer un resumen de la secuencia
model.add(GlobalAveragePooling1D())

# Capa densa para clasificación (en este caso, un ejemplo de regresión)
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


# COMPILAMOS EL MODELO DE RED NEURONAL

In [9]:
# Compilar el modelo
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Resumen del modelo
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# ENTRENAMOS EL MODELO

In [10]:
# Etiquetas de ejemplo para clasificación binaria (por ejemplo, 0 o 1)
labels = [1, 0, 1]  # Aquí se podría tener una etiqueta binaria para cada frase

# Convertir las etiquetas en un tensor
labels_tensor = tf.convert_to_tensor(labels)

# Entrenar el modelo
model.fit(padded_sequences, labels_tensor, epochs=5)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3333 - loss: 0.6993
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.3333 - loss: 0.6982
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.3333 - loss: 0.6972
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.6667 - loss: 0.6961
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.6667 - loss: 0.6950


In [11]:
# Obtener los embeddings aprendidos por la capa
embeddings = model.layers[0].get_weights()[0]
print("Embeddings learned by the model:")
print(embeddings)

Embeddings learned by the model:
[[ 0.03331297 -0.04051454 -0.00045015  0.01526017  0.04452017 -0.01072607
  -0.03915573  0.04927595]
 [-0.02392428  0.00998858 -0.02366831  0.02645104  0.02532357 -0.01040159
  -0.02028344  0.01527838]
 [ 0.03068573 -0.0456958   0.03958643  0.04984349 -0.01249641 -0.04770914
   0.02777364 -0.02988695]
 [-0.01304659 -0.04518293  0.04740504 -0.00547659 -0.0341822   0.0390751
  -0.01867951  0.0414664 ]
 [ 0.00977852  0.00572226  0.00563649  0.03241218 -0.03596599  0.02180764
  -0.0494321   0.03766774]
 [-0.01351539 -0.00476179 -0.00139007 -0.04471311 -0.0149881  -0.0438237
  -0.00901948 -0.03825232]
 [ 0.03375205  0.02286508  0.02282087 -0.02232224  0.02488363 -0.00574442
  -0.04367477 -0.04946342]
 [ 0.03591383  0.02258107 -0.04263248 -0.00238472 -0.02639717  0.00547196
  -0.05158402  0.02310579]]
